In [ ]:
!pip install transformers datasets

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [2]:
class Config:
    '''
    Class for model configuration
    vocab_size = V, 
    max_seq_length = T,
    embed_size = D,
    num_layers = 12,
    num_heads = 12,
    dropout = 0.1,
    num_experts = 4, 
    top_k = 2,
    moe_loss_weight = 0.01
    '''
    def __init__(self, vocab_size = 50257, max_seq_length = 128, embed_size = 768,
                 num_layers = 12, num_heads = 12, dropout = 0.1, num_experts = 4, 
                 top_k = 2, moe_loss_weight = 0.01):
        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.embed_size = embed_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout = dropout
        #MoE parameters
        self.num_experts = num_experts
        self.top_k = top_k
        self.moe_loss_weight = moe_loss_weight

In [3]:
class MultiHeadSelfAttention(nn.Module):
    '''
    Multihead self attention
    embed_size % num_heads = 0
    '''
    def __init__(self, config):
        super().__init__()
        # embed_size % num_heads = 0
        assert config.embed_size % config.num_heads == 0, 'Dimensions do not match'
        self.num_heads = config.num_heads
        self.head_dim = config.embed_size // config.num_heads
        # Multi head self attention
        self.W_q = nn.Linear(config.embed_size, config.embed_size) #12 W_qs, where each is shape (embed_size, head_dim)
        self.W_k = nn.Linear(config.embed_size, config.embed_size)
        self.W_v = nn.Linear(config.embed_size, config.embed_size)
        self.output = nn.Linear(config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        #lower triangular matrix for causal attention
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(config.max_seq_length, config.max_seq_length)
                      ).view(-1, 1, config.max_seq_length, config.max_seq_length))
    def forward(self, x):
        batch, seq_length, embed_dim = x.size() # MB, T, D => 16, 128, 768
        # Compute multi head Q, K, V
        # x@W_q.T => (-1, 128, 768) @ (768, 768).T => (-1, 128, 768)
        Q = self.W_q(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) 
        # (-1, 128, 768) => (batch, seq_length, num_heads, head_dim) => (batch, num_heads, seq_legth, head_dim) (-1, 12, 128, 64)
        K = self.W_k(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) 
        V = self.W_v(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) 
        attention = (Q@K.transpose(-2, -1)) / (self.head_dim ** 0.5) #(-1, 12, 128, 128)
        # (batch, num_heads, seq_legth, head_dim)@(batch, num_heads, head_dim, seq_legth) => (batch, num_heads, seq_legth, seq_length)
        # causal attention
        attention = attention.masked_fill(self.mask[:, :, :seq_length, :seq_length] == 0, float('-inf'))
        #(batch, num_heads, seq_legth, seq_length)
        attention = F.softmax(attention, dim = -1)
        #(batch, num_heads, seq_legth, seq_length)
        attention = self.dropout(attention)
        scores = attention @ V # (batch, num_heads, seq_legth, seq_length) @ ((batch, num_heads, seq_legth, head_dim)
        #(batch, num_heads, seq_legth, head_dim)
        scores = scores.transpose(1, 2).contiguous().view(batch, seq_length, embed_dim)
        #(batch, seq_legth, num_heads, head_dim) = > (batch, seq_legth, embed_dim)
        scores = self.output(scores)
        return self.dropout(scores)
        

### Mixture of Experts

In [4]:
class Expert(nn.Module):
    '''
    Feed forward neural network
    '''
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.embed_size, 4 * config.embed_size)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(4 * config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc2(self.gelu(self.fc1(x)))
        return self.dropout(x)

In [5]:
class Router(nn.Module):
    '''
    Decide what expert goes to what token
    '''
    def __init__(self, config):
        super().__init__()
        self.top_k = config.top_k
        self.gate = nn.Linear(config.embed_size, 
                              config.num_experts, 
                              bias = False)
        self.noise = nn.Linear(config.embed_size,
                               config.num_experts,
                               bias = False)
    def forward(self, x):
        # x: (num_tokens, embed_size), num_tokens = mb_size * seq_length
        logits = self.gate(x) #logits: (num_tokens, num_experts)
        if self.training:
            noise = F.softplus(self.noise(x))
            logits = logits + torch.randn_like(logits) * noise 
            # (num_tokens, num_experts)
        # softmax over experts, with what probability each token selects each expert
        probs = F.softmax(logits, dim = -1) #(num_tokens, num_experts)
        # select top_k experts
        gates, indices = torch.topk(probs, self.top_k, dim=-1)
        #gates, indices: (num_tokens, top_k)
        gates = gates / (gates.sum(dim=-1, keepdim=True))
        return gates, indices, probs

In [6]:
x = torch.randn(5, 768)
config1 = Config()
router1 = Router(config1)

In [7]:
gates, indices, probs = router1(x)

In [8]:
print(f'probs:\n {probs} \n ')
print(f'indices:\n {indices} \n')
print(f'gates:\n {gates} \n')

probs:
 tensor([[0.0833, 0.1075, 0.4306, 0.3786],
        [0.2191, 0.3249, 0.3473, 0.1087],
        [0.0838, 0.2949, 0.6099, 0.0113],
        [0.3368, 0.1747, 0.3681, 0.1204],
        [0.1465, 0.4080, 0.3283, 0.1172]], grad_fn=<SoftmaxBackward0>) 
 
indices:
 tensor([[2, 3],
        [2, 1],
        [2, 1],
        [2, 0],
        [1, 2]]) 

gates:
 tensor([[0.5322, 0.4678],
        [0.5167, 0.4833],
        [0.6740, 0.3260],
        [0.5222, 0.4778],
        [0.5541, 0.4459]], grad_fn=<DivBackward0>) 



In [9]:
experts = nn.ModuleList([Expert(config1) for _ in range(config1.num_experts)])

In [10]:
class MoE(nn.Module):
    '''
    Mixture of Experts layer, assign tokens to experts
    Also compute MoE auxiliar loss to balance expert load
    '''
    def __init__(self, config):
        super().__init__()
        self.num_experts = config.num_experts
        self.top_k = config.top_k
        self.experts = nn.ModuleList([Expert(config) for _ in range(config.num_experts)])
        self.router = Router(config)
    def forward(self, x):
        '''
        x:(batch, seq_length, embed_size)
        returns output
        '''
        batch, seq_length, embed_size = x.shape
        x_flat = x.view(-1, embed_size) #(batch*seq_length, embed_size)
        num_tokens = x_flat.shape[0]
        gates, indices, probs = self.router(x_flat)
        output = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (indices == i)
            token_mask = mask.any(dim = -1)
            if not token_mask.any():
                continue
            expert_input = x_flat[token_mask]
            expert_output = expert(expert_input)
            expert_gates = (gates * mask.float()).sum(dim = -1)
            expert_gates = expert_gates[token_mask]
                        
            output[token_mask] += expert_gates.unsqueeze(-1) * expert_output

        # MoE auxiliar loss to balance token load per expert
        expert_counts = torch.zeros(self.num_experts, device = x.device)
        expert_counts = torch.bincount(indices.flatten(), 
                                       minlength = self.num_experts).float()
        f = expert_counts / (num_tokens * self.top_k)
        p = probs.mean(dim = 0)
        expert_loss = self.num_experts * (f*p).sum()

        output = output.view(batch, seq_length, embed_size)

        return output, expert_loss

In [11]:
probs

tensor([[0.0833, 0.1075, 0.4306, 0.3786],
        [0.2191, 0.3249, 0.3473, 0.1087],
        [0.0838, 0.2949, 0.6099, 0.0113],
        [0.3368, 0.1747, 0.3681, 0.1204],
        [0.1465, 0.4080, 0.3283, 0.1172]], grad_fn=<SoftmaxBackward0>)

In [12]:
indices

tensor([[2, 3],
        [2, 1],
        [2, 1],
        [2, 0],
        [1, 2]])

In [13]:
gates

tensor([[0.5322, 0.4678],
        [0.5167, 0.4833],
        [0.6740, 0.3260],
        [0.5222, 0.4778],
        [0.5541, 0.4459]], grad_fn=<DivBackward0>)

In [14]:
x_flat = x.view(-1, config1.embed_size)

In [15]:
for i, expert in enumerate(experts):
    print(expert)

Expert(
  (fc1): Linear(in_features=768, out_features=3072, bias=True)
  (gelu): GELU(approximate='none')
  (fc2): Linear(in_features=3072, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
Expert(
  (fc1): Linear(in_features=768, out_features=3072, bias=True)
  (gelu): GELU(approximate='none')
  (fc2): Linear(in_features=3072, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
Expert(
  (fc1): Linear(in_features=768, out_features=3072, bias=True)
  (gelu): GELU(approximate='none')
  (fc2): Linear(in_features=3072, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
Expert(
  (fc1): Linear(in_features=768, out_features=3072, bias=True)
  (gelu): GELU(approximate='none')
  (fc2): Linear(in_features=3072, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [17]:
output = torch.zeros_like(x_flat)
for i, expert in enumerate(experts):
    print(f'Expert {i}')
    mask = (indices == i)
    print(f'mask {mask}')
    token_mask = mask.any(dim = -1)
    print(token_mask)
    # verify if expert i was selected by any token
    if not token_mask.any():
        continue
    expert_input = x_flat[token_mask]
    print(f'expert input: {expert_input.shape}')
    expert_output = expert(expert_input)
    print(f'expert output: {expert_output.shape}')
    expert_gates = (gates * mask.float()).sum(dim = -1)
    expert_gates = expert_gates[token_mask]
    print(f'expert_gates {expert_gates.unsqueeze(-1).shape}')
    output[token_mask] += expert_gates.unsqueeze(-1) * expert_output
    print(output.shape)
    print()
    

Expert 0
mask tensor([[False, False],
        [False, False],
        [False, False],
        [False,  True],
        [False, False]])
tensor([False, False, False,  True, False])
expert input: torch.Size([1, 768])
expert output: torch.Size([1, 768])
expert_gates torch.Size([1, 1])
torch.Size([5, 768])

Expert 1
mask tensor([[False, False],
        [False,  True],
        [False,  True],
        [False, False],
        [ True, False]])
tensor([False,  True,  True, False,  True])
expert input: torch.Size([3, 768])
expert output: torch.Size([3, 768])
expert_gates torch.Size([3, 1])
torch.Size([5, 768])

Expert 2
mask tensor([[ True, False],
        [ True, False],
        [ True, False],
        [ True, False],
        [False,  True]])
tensor([True, True, True, True, True])
expert input: torch.Size([5, 768])
expert output: torch.Size([5, 768])
expert_gates torch.Size([5, 1])
torch.Size([5, 768])

Expert 3
mask tensor([[False,  True],
        [False, False],
        [False, False],
       

### Build transformer

In [20]:
class MoETransformer(nn.Module):
    '''
    Build transformer block with MoE
    '''
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embed_size)
        self.attention = MultiHeadSelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embed_size)
        self.moe = MoE(config)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        moe_output, moe_loss = self.moe(self.norm2(x))
        x = x + moe_output
        return x, moe_loss

In [21]:
class GPT2MoE(nn.Module):
    '''
    Building GPT2 with MoE
    '''
    def __init__(self, config):
        super().__init__()
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_size) #(vocab_size, embed_size)
        self.pos_embed = nn.Embedding(config.max_seq_length, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        # MoE implementation
        self.transfomers = nn.ModuleList(
            [MoETransformer(config) for _ in range(config.num_layers)])
        self.norm1 = nn.LayerNorm(config.embed_size)

    def forward(self, input_tokens):
        batch, seq_length = input_tokens.size()
        pos = torch.arange(0, seq_length, 
                           dtype = torch.long, 
                           device = input_tokens.device).unsqueeze(0)
        
        x = self.token_embed(input_tokens) + self.pos_embed(pos) #(batch, seq_length, embed_size)
        x = self.dropout(x)

        total_loss = 0.0
        for layer in self.transfomers:
            x, moe_loss = layer(x)
            total_loss += moe_loss
             
        x = self.norm1(x)     ##(batch, seq_length, embed_size)
        logits = x @ self.token_embed.weight.t()

        return logits, total_loss
        

### Load Don Quijote Data

In [22]:
import requests
# Download Don Quijote 
url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
response = requests.get(url)
text = response.text

In [24]:
from transformers import GPT2TokenizerFast

/home/pepe/anaconda3/envs/dl-train/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
tokeniser = GPT2TokenizerFast.from_pretrained("gpt2")
tokeniser.pad_token = tokeniser.eos_token
tokens = tokeniser.encode(text)
data = torch.tensor(tokens, dtype = torch.long)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (860012 > 1024). Running this sequence through the model will result in indexing errors


In [26]:
class Quijote(Dataset):
    def __init__(self, data, seq_length):
        self.text = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.text) - self.seq_length

    def __getitem__(self, idx):
        x = self.text[idx : idx + self.seq_length]
        y = self.text[idx + 1: idx + 1 + self.seq_length]
        return x, y
        
        

In [27]:
SEQ_LENGTH = 16

In [28]:
quijote_dataset = Quijote(data, SEQ_LENGTH)

In [29]:
quijote_loader = DataLoader(quijote_dataset, batch_size = 16, shuffle= True)

### Model instantiation and training

In [30]:
config = Config(vocab_size=tokeniser.vocab_size,
                max_seq_length= SEQ_LENGTH,
                embed_size= 768,
                num_layers= 12,
                num_heads= 12,
                dropout=0.1,
                # MoE parameters
                num_experts = 4,
                top_k = 2, 
                moe_loss_weight = 0.01)

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPT2MoE(config).to(device)

In [33]:
optimiser = optim.Adam(model.parameters(), lr = 3e-4)

In [34]:
def sample(model, device, tokeniser, prompt, length = 50, 
           temperature = 1.0):
    model.eval()
    tokens = tokeniser.encode(prompt, return_tensors = 'pt').to(device)
    for _ in range(length):
        tokens_ = tokens[:, -SEQ_LENGTH:]
        with torch.no_grad():
            scores, _ = model(tokens_)

        next_token_scores = scores[:, -1, :] / temperature
        next_token = torch.multinomial(
                     F.softmax(next_token_scores, dim = -1), 
                     num_samples = 1)
        tokens = torch.cat([tokens, next_token], dim = 1)

    return tokeniser.decode(tokens[0])


In [35]:
print(sample(model, device, tokeniser, 'en un lugar de la mancha del cual', temperature=2.0))

en un lugar de la mancha del cualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualual


In [38]:
def train(model, loader, optimiser, epochs = 5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        total_ce_loss = 0.0
        total_moe_loss = 0.0
        for i, (x, y) in enumerate(loader):
            x = x.to(device)
            y = y.to(device)
            optimiser.zero_grad()
            
            scores, moe_loss = model(x)
            
            ce_loss = F.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
            loss = ce_loss + moe_loss * config.moe_loss_weight
            
            loss.backward()
            optimiser.step()

            total_loss += loss.item()
            total_ce_loss += ce_loss.item()
            total_moe_loss += moe_loss.item()

            if i % 200 == 0:
                print(f'epoch: {epoch + 1}, step:{i}, loss: {loss.item():.4f}')
                print(sample(model, device, tokeniser, 'en un lugar de la mancha del cual', temperature=0.7))
                print()
        epoch_loss = total_loss/len(loader)
        n = len(loader)
        print(f'epoch: {epoch + 1}, loss: {epoch_loss:.4f}, '
              f'CE loss: {total_ce_loss/n:.4f}, '
              f'MoE loss: {total_moe_loss/n: .4f}')
        print()
    

In [39]:
train(model, quijote_loader, optimiser, 5)

epoch: 1, step:0, loss: 249.0335
en un lugar de la mancha del cualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualual

epoch: 1, step:200, loss: 17.0531
en un lugar de la mancha del cualualualualualualualualadasadasadasadasadasadasadasadasadasadasadasadasadasadasadasLLLo de pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas pas

epoch: 1, step:400, loss: 8.7721
en un lugar de la mancha del cual nas
leerien señor, señien era; y v vuestra su él; y de p escón
ichichichí, señora era era era era era era era, en

epoch: 1, step:600, loss: 7.2771
en un lugar de la mancha del cual que de los dondeó la cual los � � � � � � � � � � � � �ió de uno donde—, sin duda vi vi vios os, señora de mi mi mi señ

epoch: 1, step:800, loss: 5.8881
en un lugar de la mancha del cual de más de lo que los tr podrado y
fada y dijo, y dijo y cigo y el también y dijo de su tenga y c

KeyboardInterrupt: 

In [71]:
torch.save({
    'config': config.__dict__,
    'state_dict': model.state_dict()
}, "gpt2_model_and_config.pth")


In [68]:
torch.save(model.state_dict(), "gpt2_260725_weights.pth")

In [70]:
tokeniser.save_pretrained("my_tokenizer/")

('my_tokenizer/tokenizer_config.json',
 'my_tokenizer/special_tokens_map.json',
 'my_tokenizer/vocab.json',
 'my_tokenizer/merges.txt',
 'my_tokenizer/added_tokens.json',
 'my_tokenizer/tokenizer.json')

In [ ]:
model = GPT2Model(config)
model.load_state_dict(torch.load("gpt2_260725_weights.pth"))
